<center><h1> Customer Segmentation </h1></center>

<a id="import"></a>
# 1️⃣ㅣImport Libraries

In [ ]:
!pip install kneed

In [ ]:
#Import libraries
#loading dataset
import pandas as pd
import numpy as np

#visualisation
import matplotlib.pyplot as plt
%matplotlib inline
import plotly.express as px
import seaborn as sns
from kneed import KneeLocator

# data modeling
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import ward,dendrogram,linkage
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
from sklearn.cluster import AgglomerativeClustering
from sklearn.mixture import GaussianMixture

# Model performance
from sklearn.preprocessing import  StandardScaler
from sklearn import metrics
from sklearn.metrics import silhouette_score
from tqdm import tqdm
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import davies_bouldin_score
from sklearn.decomposition import PCA

#warnings
import warnings
warnings.simplefilter(action='ignore')

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

<a id="data"></a>
# 2️⃣ㅣLoad data💾 

In [ ]:
# reading data into dataframe
df=pd.read_csv('kaggle/input/customer-segmentation/Customer_Data (1).csv')

df

In [ ]:
df.shape

- CUST_ID: Credit card holder ID
- BALANCE: Monthly average balance (based on daily balance averages)
- BALANCE_FREQUENCY: Ratio of last 12 months with balance.How frequently the Balance is updated, score between 0 and 1 (1 = frequently updated, 0 = not frequently updated)
- PURCHASES: Total purchase amount spent during last 12 months
- ONEOFF_PURCHASES: Total amount of one-off purchases
- INSTALLMENTS_PURCHASES: Total amount of installment purchases
- CASH_ADVANCE: Total cash-advance amount
- PURCHASES_ FREQUENCY: Frequency of purchases (Percent of months with at least one purchase).How frequently the Purchases are being made score between 0 and 1 (1 = frequently purchased, 0 = not frequently purchased)
- ONEOFF_PURCHASES_FREQUENCY: Frequency of one-off-purchases.How frequently Purchases are happening in one-go (1 = frequently purchased, 0 = not frequently purchased) 
- PURCHASES_INSTALLMENTS_FREQUENCY: Frequency of installment purchases.How frequently purchases in installments are being done (1 = frequently done, 0 = not frequently done)
- CASH_ADVANCE_ FREQUENCY: Cash-Advance frequency
- AVERAGE_PURCHASE_TRX: Average amount per purchase transaction
- CASH_ADVANCE_TRX: Average amount per cash-advance transaction
- PURCHASES_TRX: Average amount per purchase transaction
- CREDIT_LIMIT: Credit limit
- PAYMENTS: Total payments (due amount paid by the customer to decrease their statement balance) in the period
- MINIMUM_PAYMENTS: Total minimum payments due in the period.
- PRC_FULL_PAYMEN: Percentage of months with full payment of the due statement balance
- TENURE: Number of months as a customer

In [ ]:
# lists name of columns
df.columns

### <a id="eda"></a>
# 3️⃣ ㅣData Preprocessing

In [ ]:
#to view some basic statistical details of train data
df.describe()

In [ ]:
# checking the number of rows and columns in train dataset
rows,col=df.shape
print ('Row:' , rows,'\nColumns:',col)

In [ ]:
#getting some information about the datafram
df.info()

In [ ]:
## printing total numbers of Unique value in the dataframe. 
df.nunique()

In [ ]:
# Find the total number of missing values in the dataframe
df.isnull().sum()

- we can see we have 1 null value in CREDIT_LIMIT and 313 in MINIMUM_PAYMENTS

In [ ]:
missing_count=df.isnull().sum() #the count of missing values
value_count= df.isnull().count()#the count of all values
missing_percentage= round(missing_count/value_count*100,2)#the percentage of missing values
missing_df= pd.DataFrame({'count': missing_count,'percentage':missing_percentage })#create a dataframe
print(missing_df)

In [ ]:
barchart=missing_df.plot.bar(y= 'percentage')
for index, percentage in enumerate(missing_percentage):
    barchart.text(index,percentage, str(percentage)+"%")

In [ ]:

# checking the value which is Null for Credit Limit
df[df['CREDIT_LIMIT'].isnull()]

In [ ]:
# dropping off the missing value for Credit Limit

df = df.drop(5203)
# resetting the index after dropping the record:
df = df.reset_index(drop=True)

In [ ]:
df[['PAYMENTS', 'MINIMUM_PAYMENTS']][df['MINIMUM_PAYMENTS'].isna()]

- It is pretty obvious that customers who haven't done any payments wouldn't have minimum payments also
- Hence, if PAYMENTS = 0, MINIMUM_PAYMENTS = 0

In [ ]:
# Looking at PAYMENTS that's above MINIMUM_PAYMENTS

print('Shape:', df[['PAYMENTS', 'MINIMUM_PAYMENTS']][(df['MINIMUM_PAYMENTS'].notna()) & (df['PAYMENTS'] > df['MINIMUM_PAYMENTS'])].shape)
df[['PAYMENTS', 'MINIMUM_PAYMENTS']][(df['MINIMUM_PAYMENTS'].notna()) & (df['PAYMENTS'] > df['MINIMUM_PAYMENTS'])].head()

In [ ]:
# Looking at PAYMENTS that's below MINIMUM_PAYMENTS

print('Shape:', df[['PAYMENTS', 'MINIMUM_PAYMENTS']][(df['MINIMUM_PAYMENTS'].notna()) & (df['PAYMENTS'] < df['MINIMUM_PAYMENTS'])].shape)
df[['PAYMENTS', 'MINIMUM_PAYMENTS']][(df['MINIMUM_PAYMENTS'].notna()) & (df['PAYMENTS'] < df['MINIMUM_PAYMENTS'])].head()

- Logically, payments should be done if PAYMENTS above MINIMUM_PAYMENTS. Which is true for 6272 customers.
- But on the other hand, we got 2364 customers who did the PAYMENTS below MINIMUM_PAYMENTS which lead to invalid values. But we would like to leave it that way.
- Hence, we fill the missing values by the mean of PAYMENTS
- if PAYMENTS is less than MINIMUM_PAYMENTS, the missing values will be filled by the correspond PAYMENTS

In [ ]:
minpay = df['MINIMUM_PAYMENTS'].copy() # make a copy of MINIMUM_PAYMENTS
payments_mean = np.mean(df['PAYMENTS']) # take the mean value of PAYMENTS

i = 0
for payments, minpayments in zip(df['PAYMENTS'], df['MINIMUM_PAYMENTS'].isna()):
    if (payments == 0) and (minpayments == True):
        minpay[i] = 0
    elif (0 < payments < payments_mean) and (minpayments == True): 
        minpay[i] = payments
    elif minpayments == True: 

        minpay[i] = payments_mean
    i += 1
    
df['MINIMUM_PAYMENTS'] = minpay.copy()


In [ ]:
# Now again check the missing values.

df.isnull().any()

In [ ]:
#lets see if we have any duplicated entries and the result shows that all entries are unique
df.duplicated().sum()

In [ ]:
#drop the CUST_ID column which doesn't provide any info .
df.drop(columns='CUST_ID' , axis=1 , inplace=True)

In [ ]:
print('Number of columns={}'.format(len(df.columns)))

#### Monthly_avg_purchase

In [ ]:
df['Monthly_avg_purchase']=df['PURCHASES']/df['TENURE']

In [ ]:
print(df['Monthly_avg_purchase'].head(),'\n ',
df['TENURE'].head(),'\n', df['PURCHASES'].head())

#### Monthly_cash_advance Amount

In [ ]:
df['Monthly_cash_advance']=df['CASH_ADVANCE']/df['TENURE']

In [ ]:
df[df['ONEOFF_PURCHASES']==0]['ONEOFF_PURCHASES'].count()

### <a id="eda"></a>
# 4️⃣ㅣExploratory Data Analysis (EDA)📊

In [ ]:
df.boxplot(rot=90 , figsize=(30,20))

In [ ]:
df.hist(figsize=(10,8))
plt.tight_layout()

In [ ]:
sns.pairplot(df)

Observation:
- 1- As the credit limit increase, the balance also increases hence a linear relationship
- 2- As the number of purchases increases, the number of "cash in advance" transactions decreases
- 3- As the credit balance is low, the purchases, oneoffpurchases and installments purchases are less. Thus validating our assumption from univariate analysis
- 4- Purchases, oneoffpurchases and installment purchases are all related linearly
- 5- As the credit balance is low, the "cash in advance" transactions are less

In [ ]:
#we are going to use dist_plot which is a combination of "hist" function in matplotlib and "KDE" in seaborn
palette_color=sns.color_palette('bright')
plt.figure(figsize=(10,20))
for i in range (len(df.columns)):
    plt.subplot(10,2,i+1)
    sns.histplot(df[df.columns[i]] ,kde=True)
    plt.title(df.columns[i])
    plt.tight_layout()

Observation:
- 1- Most credit card holders have low credit limit and maintain credit balance below 7500
- 2- Variable such as Purchases, OneOffPurchases, installmentpurchases and cash advances also follow the same trend as     credit balance. They could all be related. That is as the credit balance is low, the purchases are also low and so on
- 3- Most people either don't purchase anything or they purchase very frequently
- 4- People who purchase in installments is more than people who purchase in one-go
- 5- In the last 6 months, most people have made total payments below 10000, with the minimum payments below 5000
- 6- Finally, most of the credit card holders own a card for more than 12 months

<a id="modeling"></a>
# 5️⃣ ㅣCorrelation

In [ ]:
plt.figure(figsize = (18,10))
sns.heatmap(df.corr(), cmap= 'plasma',annot = True , fmt='.2f' )

Balance has a higher level of correlation with Cash Advance, Cash Advance Frequency and Credit Limit. Payments variable has a high correletion with Purchases and one off Purchases. Tenure has a negative correlation with Cash Advance and Cash Advance Frequency variables.
"PURACHASES_INSTALLMENTS_FREQUENCY" has strong positive correlation with "PURCHASE FREQUENCY",
            "BALANCE" has a strong negative correlation with "PRC_FULL_PAYMENT",
            "TENURE" has almost no correlation with any (No linear relationship)

In [ ]:
sns.jointplot(x='PURCHASES' , y='ONEOFF_PURCHASES', data=df)
plt.show()

In [ ]:
sns.jointplot(x='PURCHASES', y='PURCHASES_FREQUENCY', data=df)

Customers with more range of frequency have higher amount of purchases

In [ ]:
sns.jointplot(x='PURCHASES', y='CASH_ADVANCE', data=df)

In [ ]:
sns.relplot(x='PURCHASES',y='CREDIT_LIMIT',data=df)

- high purchases in the range 0-10000
- Credit limit more in the range 0-20000

In [ ]:
sns.jointplot(x='CREDIT_LIMIT', y='BALANCE', data=df)

- CREDIT_LIMIT & BALANCE are in a linear relation with each other ie. if a customer's balance increases, his/her credit limit shall increase
- Most customers lie in and under 15000 credit limit and 10K balance
others can be considered as premium customers

In [ ]:
sns.jointplot(x='BALANCE_FREQUENCY', y='BALANCE', data=df)

In [ ]:
sns.relplot(x='MINIMUM_PAYMENTS',y='BALANCE',data=df)

In [ ]:
sns.jointplot(x='TENURE', y='PURCHASES', data=df)

In [ ]:
sns.relplot(x='TENURE',y='PURCHASES_FREQUENCY',data=df)

In [ ]:
sns.relplot(x='TENURE',y='CASH_ADVANCE_FREQUENCY',data=df)

- Old customers(with tenure=12) have cash adv freq less than one
- New customers(tenure 6-11) take cash advance more frequently


In [ ]:
sns.jointplot(x='TENURE', y='BALANCE', data=df)

People with 12 as tenure tend to have more balance in their account - Customer Loyalty

In [ ]:
sns.relplot(x='TENURE',y='PAYMENTS',data=df)

more tenure more payment

In [ ]:
sns.relplot(x='TENURE',y='MINIMUM_PAYMENTS',data=df)

 More the tenure more is minimum payment

In [ ]:
plt.figure(figsize=(10,5))
px.scatter(df, x='TENURE', y='CREDIT_LIMIT', color='TENURE')

With increase in TENURE, CREDIT_LIMIT increases

In [ ]:
plt.figure(figsize = (20,10))
sns.barplot(x = df['TENURE'].unique(),
            y = df['TENURE'].value_counts(dropna = False),
            palette = "viridis")
plt.xlabel('Tenure',fontsize=15)
plt.ylabel('Customers count',fontsize=15)
plt.title('Customer count in particular Tenure',fontsize=20)
plt.show()

In [ ]:
(1e2*df['TENURE'].value_counts().sort_index()/len(df)).plot(kind='barh')
plt.title('Tenure Distribution')
plt.xlabel('% Distribution');

In [ ]:
sns.boxplot(x="TENURE", y="BALANCE", data=df)
plt.ylim(-10**3, 10**4)
plt.title('Balance distribution with Tenure');

In [ ]:
ax = df[['BALANCE_FREQUENCY', 'PURCHASES_FREQUENCY',
         'ONEOFF_PURCHASES_FREQUENCY', 'PURCHASES_INSTALLMENTS_FREQUENCY',
         'CASH_ADVANCE_FREQUENCY','PRC_FULL_PAYMENT']].plot.kde(figsize=(8,5), bw_method=3) #,ind=[0, 2, 3,4]

In [ ]:
fig= plt.subplots(nrows=7 , ncols=3 , figsize=(20,20))
for i in range (len(df.columns)):
    plt.subplot(7,3,i+1)
    ax=  sns.boxplot(df[df.columns[i]])
    plt.title(df.columns[i])
    plt.tight_layout()

It looks like we have significant problem with outliers.

 Outlier detection model selection
- distribution is not normal
- distribution is highly skewed
- we have huge outliers
Isolation Forest does not assume normal distribution and is able to detect outliers at a multi-dimensional level. Isolation Forest is also computationally efficient. The algorithm is based on the principle that anomalies are observations that are few and different, this should make them easier to identify. That's why I choose Isolation Forest.

####  Isolation Forest

Isolation Forest(IF) is similar to Random Forest and it is build based on decision trees. There are no pre-defined labels here. It's an unsupervised learning algorithm that identifies anomaly by isolating outliers in the data.

The Isolation Forest (iForest) algorithm was initially proposed by Fei Tony Liu, Kai Ming Ting and Zhi-Hua Zhou in 2008. The authors took advantage of two quantitative properties of anomalous data points in a sample:

 - 1- Few - they are the minority consisting of fewer instances and
 - 2- Different - they have attribute-values that are very different from those of normal instances
Since the Isolation Forest algorithm is based on the principle that anomalies are observations that are few and different, this should make them easier to identify.

Isolation Forest does not assume normal distribution and is able to detect outliers at a multi-dimensional level. Isolation Forest is computationally efficient: the algorithm has a linear time complexity with a low constant and a low memory requirement. Therefore, it scales well to large data sets.

- https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html

In [ ]:
from sklearn.ensemble import IsolationForest
df1 = df.copy()

In [ ]:
# Model building
model=IsolationForest(n_estimators=150, max_samples='auto', contamination=float(0.1), max_features=1.0)
model.fit(df1)

In [ ]:
# Adding 'scores' and 'anomaly' colums to df
scores=model.decision_function(df1)
anomaly=model.predict(df1)

df1['scores']=scores
df1['anomaly']=anomaly

anomaly = df1.loc[df1['anomaly']==-1]
anomaly_index = list(anomaly.index)
print('Total number of outliers is:', len(anomaly))

In [ ]:
# dropping outliers
df1 = df1.drop(anomaly_index, axis = 0).reset_index(drop=True)

In [ ]:
fig= plt.subplots(nrows=7 , ncols=3 , figsize=(20,20))
for i in range (len(df1.columns)):
    plt.subplot(7,3,i+1)
    ax=  sns.boxplot(df1[df1.columns[i]])
    plt.title(df1.columns[i])
    plt.tight_layout()

In [ ]:
#drop the 'scores','anomaly' columns which doesn't provide any info .
df1.drop(columns='scores' , axis=1 , inplace=True)
df1.drop(columns='anomaly' , axis=1 , inplace=True)

<a id="summary"></a>
# 6️⃣ㅣ Model Building

#### 1.  Standardrizing data

In [ ]:
scaler= StandardScaler()
scaled_features=scaler.fit_transform(df1)
scaled_df=pd.DataFrame(scaler.fit_transform(df1), columns = df1.columns)
scaled_df

### 2. Determining The Optimal Number Of Clusters
 Selecting optimal number of clusters is key to applying clustering algorithm to the dataset, such as k-means clustering, which requires the user to specify the number of clusters k to be generated. This is a somewhat arbitrary procedure, one of the weakest aspects of performing cluster analysis.

 The major difference between elbow and silhouette method is that elbow only calculates the euclidean distance whereas silhouette takes into account variables such as variance, skewness, high-low differences, etc.

 Both the Elbow method / SSE Plot and the Silhouette method can be used interchangeably based on the details presented by the plots.

### 2.1 Elbow Method

Elbow method is a very popular method to calculate optimal number of clusters for a given problem. Within clusters the sum of square distance is calculated and plotted against the number of clusters. The elbow point in plot is selected as optimal number of clusters for given problem.

In [ ]:
kmeans_set={"init":"random","n_init":10,"max_iter":300,"random_state":42}

In [ ]:
cluster_range = range( 1, 21 )
cluster_errors=[]
for k in cluster_range:
    kmeans= KMeans(n_clusters=k, **kmeans_set) #** open dictionry
    kmeans.fit(scaled_features)
    cluster_errors.append( kmeans.inertia_) 

In [ ]:
clusters_df = pd.DataFrame( { "num_clusters":cluster_range, "cluster_errors": cluster_errors } )

#clusters_df[0:21]
clusters_df.head(10)

In [ ]:
plt.style.use("fivethirtyeight")
plt.plot(range(1,21),cluster_errors)
plt.xticks(range(1,21))
plt.xlabel('number of clusters')
plt.ylabel('inertia')
plt.show()

In [ ]:
from yellowbrick.cluster import KElbowVisualizer
model = KMeans(n_clusters=k, **kmeans_set)
# k is range of number of clusters.
visualizer = KElbowVisualizer(model, k=(2,30), timings= True)
visualizer.fit(scaled_features)        # Fit data to visualizer
visualizer.show()        # Finalize and render figure

In [ ]:
# Now we are going to implement Elbow method to final optimal number of clusters
k1=KneeLocator(range(1,21),cluster_errors , curve='convex', direction= 'decreasing')
k1.elbow

In [ ]:
plt.style.use("fivethirtyeight")
plt.plot(range(1,21),cluster_errors)
plt.xticks(range(1,21))
plt.xlabel('number of clusters')
plt.ylabel('List')
plt.axvline(x=k1.elbow, color='b', label= 'axvline-full height', ls= '--')
plt.show()

## 2.2 Silhouette Coefficient Method:
 Average silhouette approach measures the quality of a clustering. That is, it determines how well each object lies within its cluster. A high average silhouette width indicates a good clustering. The optimal number of clusters k is the one that maximize the average silhouette over a range of possible values for k.
 
 The silhouette score falls within the range [-1, 1].

 The silhouette score of 1 means that the clusters are very dense and nicely separated. The score of 0 means that clusters are overlapping. The score of less than 0 means that data belonging to clusters may be wrong/incorrect.

- elbow method: k=5 , k=10 
- silhouette coefficiennt: k=3

In [ ]:
#let's try silhouette_score for k=5,10,3

# Instantiate the KMeans for 5 clusters
km = KMeans(n_clusters=5, random_state=42)
# Fit the KMeans model
km.fit_predict(scaled_features)
# Calculate Silhoutte Score
score = silhouette_score(scaled_features, km.labels_, metric='euclidean')
# Print the score
print('Silhouetter Average Score: %.3f' % score)

In [ ]:

# Instantiate the KMeans for 10 clusters
km = KMeans(n_clusters=10, random_state=42)
# Fit the KMeans model
km.fit_predict(scaled_features)
# Calculate Silhoutte Score
score = silhouette_score(scaled_features, km.labels_, metric='euclidean')
# Print the score
print('Silhouetter Average Score: %.3f' % score)

In [ ]:

# Instantiate the KMeans for 3 clusters
km = KMeans(n_clusters=3, random_state=42)
# Fit the KMeans model
km.fit_predict(scaled_features)
# Calculate Silhoutte Score
score = silhouette_score(scaled_features, km.labels_, metric='euclidean')
# Print the score
print('Silhouetter Average Score: %.3f' % score)

In [ ]:
# Instantiate the KMeans for 2 clusters
km = KMeans(n_clusters=2, random_state=42)
# Fit the KMeans model
km.fit_predict(scaled_features)
# Calculate Silhoutte Score
score = silhouette_score(scaled_features, km.labels_, metric='euclidean')
# Print the score
print('Silhouetter Average Score: %.3f' % score)

#### the best silhouette_score k=3

In [ ]:
silhouette_coefficients =[]
for k in range(2,21):#1 is the worse
    kmeans=KMeans(n_clusters=k, **kmeans_set)
    kmeans.fit(scaled_features)
    score= silhouette_score(scaled_features, kmeans.labels_)
    silhouette_coefficients.append(score)

In [ ]:
silhouette_coefficients

In [ ]:
plt.style.use("fivethirtyeight")
plt.plot(range(2,21),silhouette_coefficients)
plt.xticks(range(2,21))
plt.xlabel('number of clusters')
plt.ylabel('silhouette coefficients')
plt.show()

In [ ]:
def evaluate_metrics(df1, min_clust=2, max_clust=10, rand_state=42):
    inertias = []
    silhouette = []
    ch_score = []
    db_score = []
    for n_clust in range(min_clust, max_clust):
        kmeans = KMeans(n_clusters=n_clust, random_state=rand_state)
        y_label = kmeans.fit_predict(scaled_features)
        inertias.append(kmeans.inertia_)
        silhouette.append(silhouette_score(scaled_features, y_label))
        ch_score.append(calinski_harabasz_score(scaled_features, y_label))
        db_score.append(davies_bouldin_score(scaled_features, y_label))        

    fig, ax = plt.subplots(2, 2, figsize=(15, 10))
    ax[0][0].plot(range(min_clust, max_clust), inertias, '-x', linewidth=2)
    ax[0][0].set_xlabel('No. of clusters')
    ax[0][0].set_ylabel('Inertia')
    
    ax[0][1].plot(range(min_clust, max_clust), silhouette, '-x', linewidth=2)
    ax[0][1].set_xlabel('No. of clusters')
    ax[0][1].set_ylabel('Silhouette Score')
    
    ax[1][0].plot(range(min_clust, max_clust), ch_score, '-x', linewidth=2)
    ax[1][0].set_xlabel('No. of clusters')
    ax[1][0].set_ylabel('Calinski Harabasz Score')
    
    ax[1][1].plot(range(min_clust, max_clust), db_score, '-x', linewidth=2)
    ax[1][1].set_xlabel('No. of clusters')
    ax[1][1].set_ylabel('Davies Bouldin Score')
    fig.suptitle('Metrics to evaluate the number of clusters')
    plt.show()

In [ ]:
evaluate_metrics(scaled_features, min_clust=2, max_clust=15, rand_state=42)

So based on the plots above we conclude that 3 clusters are the best for k-mean modeling

In [ ]:
silhouette_scores = []

for n_cluster in range(2, 13):
    silhouette_scores.append( 
        silhouette_score(scaled_features, KMeans(n_clusters = n_cluster).fit_predict(scaled_features))) 
    
# Plotting a bar graph to compare the results 
k = [2, 3, 4, 5, 6,7,8,9,10,11,12] 

plt.bar(k, silhouette_scores) 
plt.xlabel('Number of clusters', fontsize = 10) 
plt.ylabel('Silhouette Score', fontsize = 10) 
plt.show()
#confirming number of clusters

Note : Highest silhouette score is k = 3.

In [ ]:
from yellowbrick.cluster import SilhouetteVisualizer
# Yellowbrick extends the Scikit-Learn API to make model selection and hyperparameter tuning easier.
# You can find the code to simply create Silhouette visualisation for K-Means clusters with n_cluster as 2, 3, 4, 5, 6, 7,8,9,10,11,12,13 below.

fig, ax = plt.subplots(6, 2, figsize=(20,15))
fig.suptitle('Silhouette Analysis for 2-13 Clusters', size = 18)
plt.tight_layout()

for i in [2, 3, 4, 5, 6, 7,8,9,10,11,12,13]:
    '''
    Create KMeans instance for different number of clusters
    '''
    km = KMeans(n_clusters=i, init='k-means++', n_init=10, max_iter=100, random_state=42)
    q, mod = divmod(i, 2)
    '''
    Create SilhouetteVisualizer instance with KMeans instance
    Fit the visualizer
    '''
    visualizer = SilhouetteVisualizer(km, colors='yellowbrick', ax=ax[q-1][mod])
    visualizer.fit(scaled_features)

- Having clusters with negative silhouette scores for all values of k in the above plots shows that kmeans has not performed well in clustering, but in any case, it seems that k=3 is the best number of clusters

Conclusion:
- 3 Clusters is chosen,because the highest value in Silhouetter Score and calinski_harabasz_score are positioned at 3th cluster. 

### 2.3 Dendograms 
 This technique is specific to the agglomerative hierarchical method of clustering. The method starts by considering each point as a separate cluster and starts joining points to clusters in a hierarchical fashion based on their distances. To get the optimal number of clusters for hierarchical clustering, we make use a dendrogram which is tree-like chart that shows the sequences of merges or splits of clusters.

In [ ]:
distance = linkage(scaled_features,'ward')

In [ ]:
plt.figure(figsize=(20,10))
plt.title("Hierarchical Clustering Dendogram")
plt.xlabel("Index")
plt.ylabel("Ward's Distance")
dendrogram(distance, leaf_rotation=90, leaf_font_size=9);
plt.axhline(98, c='k')

### 2 K-Means Algorithm

In [ ]:
# model with 3 clusters
kmeans_3=KMeans(n_clusters=3)
y_kmeans = kmeans_3.fit_predict(scaled_features)
y_kmeans


In [ ]:
cluster_centers = pd.DataFrame(data = kmeans.cluster_centers_, columns= [df1.columns])
cluster_centers

In [ ]:
kmeans_3.labels_


In [ ]:
pd.Series(kmeans_3.labels_).value_counts()

In [ ]:
df1['kmeans_3.labels_']=kmeans_3.labels_

In [ ]:
# Create a list of all the features
features = ['BALANCE', 'BALANCE_FREQUENCY', 'PURCHASES',
       'ONEOFF_PURCHASES', 'INSTALLMENTS_PURCHASES', 'CASH_ADVANCE',
       'PURCHASES_FREQUENCY', 'ONEOFF_PURCHASES_FREQUENCY',
       'PURCHASES_INSTALLMENTS_FREQUENCY', 'CASH_ADVANCE_FREQUENCY',
       'CASH_ADVANCE_TRX', 'PURCHASES_TRX', 'CREDIT_LIMIT', 'PAYMENTS',
       'MINIMUM_PAYMENTS', 'PRC_FULL_PAYMENT', 'TENURE']

In [ ]:
# Plot the average feature value for each cluster
describe_clusters = scaled_df.groupby(df1['kmeans_3.labels_']).mean()
describe_clusters


In [ ]:
for i in features:
    
    sns.barplot(x = describe_clusters.index
               , y = i
               , data = describe_clusters)
    plt.title('Mean ' + str(i) + ' by Cluster')
    plt.show()

In [ ]:
df1.head(10)

In [ ]:
# Plot the histogram of various clusters
for i in df1.columns:
    plt.figure(figsize = (35, 5))
    for j in range(3):
        plt.subplot(1,3,j+1)
        ax = df1[df1['kmeans_3.labels_'] == j]
        ax[i].hist(bins=20)
        plt.title('{}    \nCLUSTER {} '.format(i,j))
         
plt.show()

### 2.1 Principal componenet Analysis (PCA)

In [ ]:
#Lets convert our data to only 2D using PCA
pca = PCA(n_components=2)
pca_components = pca.fit_transform(scaled_features)
pca_components

In [ ]:
# create a dataframe of these two componenets
pca_df = pd.DataFrame(data = pca_components, columns = ['pca1', 'pca2'])
pca_df.head()

In [ ]:
# Concatenate the clusters labels to the dataframe
pca_df = pd.concat([pca_df,pd.DataFrame({'cluster':kmeans_3.labels_})], axis = 1)
pca_df.head()

In [ ]:
# Visualize the clusters in 2-axes plane
ax = sns.scatterplot(x="pca1", y="pca2", hue = "cluster", data = pca_df, palette =['red','green','blue'])
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=3,init= "random", random_state = 42).fit(df1)
centroids = kmeans.cluster_centers_
plt.figure(figsize=(10,6))

df_kmean = df1.copy()
df_kmean['cluster'] = kmeans.labels_
sns.relplot(data = df_kmean ,x='BALANCE', y='PURCHASES', hue="kmeans_3.labels_", palette=['red','green','yellow'] ,kind='scatter', aspect = 11.7/8.27)
plt.scatter(centroids[:, 0], centroids[:,1], c='blue', s=50)
plt.xlabel("BALANCE",fontsize=15)
plt.ylabel("PURCHASES",fontsize=15)

In [ ]:
kmeans = KMeans(n_clusters=3,init= "random", random_state = 42).fit(df1)
centroids = kmeans.cluster_centers_
plt.figure(figsize=(10,6))

df_kmean = df1.copy()
df_kmean['cluster'] = kmeans.labels_
sns.relplot(data = df_kmean ,x='PURCHASES', y='CASH_ADVANCE', hue="kmeans_3.labels_", palette=['red','green','yellow'] ,kind='scatter', aspect = 11.7/8.27)
plt.scatter(centroids[:, 0], centroids[:,1], c='blue', s=50)
plt.xlabel("PURCHASES",fontsize=15)
plt.ylabel("CASH_ADVANCE",fontsize=15)

In [ ]:
best_cols = ["BALANCE", "PURCHASES", "CASH_ADVANCE","CREDIT_LIMIT", "PAYMENTS", "MINIMUM_PAYMENTS","kmeans_3.labels_"]
sns.pairplot(df1[ best_cols ], hue="kmeans_3.labels_")

In [ ]:
(df1[["BALANCE", "PURCHASES", "CASH_ADVANCE","CREDIT_LIMIT", "PAYMENTS", "MINIMUM_PAYMENTS","kmeans_3.labels_"]]
 .groupby("kmeans_3.labels_").mean().plot.bar(figsize=(15, 5)))
plt.title('Purchase Behavior of various segments')
plt.xlabel('SEGMENTS');

In [ ]:
(df1[['PURCHASES_FREQUENCY', 'ONEOFF_PURCHASES_FREQUENCY', 'PURCHASES_INSTALLMENTS_FREQUENCY', 'CASH_ADVANCE_FREQUENCY',"kmeans_3.labels_"]]
 .groupby("kmeans_3.labels_").mean().plot.bar(figsize=(15, 5)))
plt.title('Frequency behavior of various segments')
plt.xlabel('SEGMENTS');

### 3. Hierarchical clustering
 Hierarchical clustering (also called hierarchical cluster analysis or HCA) is a method of cluster analysis that seeks to build a hierarchy of clusters. It's used to group objects in clusters based on how similar they are to each other. In general, the merges and splits are determined in a greedy manner. The results of hierarchical clustering are usually presented in a dendrogram.

In [ ]:
# Training model
df_AgglomerativeC = scaled_df.copy()
AgglomerativeC = AgglomerativeClustering(n_clusters=2, metric = 'euclidean', linkage = 'ward')
y_AgglomerativeC = AgglomerativeC.fit_predict(df_AgglomerativeC)


In [ ]:
# We called the df, that's why we need to refer to previous df to add cluster numbers

# Checking number of items in clusters and creating 'Cluster' column
df_AgglomerativeC['Cluster'] = y_AgglomerativeC
df_AgglomerativeC['Cluster'].value_counts()

In [ ]:
def evaluate_metrics(df1, min_clust=2, max_clust=10, rand_state=42):

    silhouette = []
    ch_score = []
    db_score = []
    for n_clust in range(min_clust, max_clust):
        AgglomerativeC = AgglomerativeClustering(n_clusters=n_clust, metric = 'euclidean', linkage = 'ward')
        y_label = AgglomerativeC.fit_predict(df_AgglomerativeC)
        
        silhouette.append(silhouette_score(scaled_features, y_label))
        ch_score.append(calinski_harabasz_score(scaled_features, y_label))
        db_score.append(davies_bouldin_score(scaled_features, y_label))        

    
    fig, ax = plt.subplots(2, 2, figsize=(15, 10))
    ax[0][0].plot(range(min_clust, max_clust), silhouette, '-x', linewidth=2)
    ax[0][0].set_xlabel('No. of clusters')
    ax[0][0].set_ylabel('Silhouette Score')
    
    ax[0][1].plot(range(min_clust, max_clust), ch_score, '-x', linewidth=2)
    ax[0][1].set_xlabel('No. of clusters')
    ax[0][1].set_ylabel('Calinski Harabasz Score')
    
    ax[1][0].plot(range(min_clust, max_clust), db_score, '-x', linewidth=2)
    ax[1][0].set_xlabel('No. of clusters')
    ax[1][0].set_ylabel('Davies Bouldin Score')
    fig.suptitle('Metrics to evaluate the number of clusters')
    plt.show()

In [ ]:
evaluate_metrics(scaled_features, min_clust=2, max_clust=15, rand_state=42)

In [ ]:
silhouette_scores = []

for n_cluster in range(2, 13):
    silhouette_scores.append( 
        silhouette_score(scaled_features, AgglomerativeClustering(n_clusters = n_cluster).fit_predict(df_AgglomerativeC))) 
    
# Plotting a bar graph to compare the results 
k = [2, 3, 4, 5, 6,7,8,9,10,11,12] 

plt.bar(k, silhouette_scores) 
plt.xlabel('Number of clusters', fontsize = 10) 
plt.ylabel('Silhouette Score', fontsize = 10) 
plt.show()
#confirming number of clusters

Note : Highest silhouette score is k = 2.

In [ ]:
# Instantiate the AgglomerativeC for 2 clusters
AgglomerativeC = AgglomerativeClustering(n_clusters=2, metric = 'euclidean', linkage = 'ward')
# Fit the AgglomerativeC model
AgglomerativeC.fit_predict(df_AgglomerativeC)
# Calculate Silhoutte Score
score = silhouette_score(scaled_features, AgglomerativeC.labels_, metric='euclidean')
# Print the score
print('Silhouetter Average Score: %.3f' % score)

In [ ]:
# Instantiate the AgglomerativeC for 3 clusters
AgglomerativeC = AgglomerativeClustering(n_clusters=3, metric = 'euclidean', linkage = 'ward')
# Fit the AgglomerativeC model
AgglomerativeC.fit_predict(df_AgglomerativeC)
# Calculate Silhoutte Score
score = silhouette_score(scaled_features, AgglomerativeC.labels_, metric='euclidean')
# Print the score
print('Silhouetter Average Score: %.3f' % score)

In [ ]:
AgglomerativeC = AgglomerativeClustering(n_clusters=2, metric = 'euclidean', linkage = 'ward')
y_AgglomerativeC = AgglomerativeC.fit_predict(df_AgglomerativeC)

In [ ]:
# Checking number of items in clusters and creating 'Cluster' column
df_AgglomerativeC['Cluster'] = y_AgglomerativeC
df_AgglomerativeC['Cluster'].value_counts()

In [ ]:
describe_clusters1 = scaled_df.groupby(df_AgglomerativeC['Cluster']).mean()
describe_clusters1

In [ ]:
for i in features:
    
    sns.barplot(x = describe_clusters1.index
               , y = i
               , data = describe_clusters1)
    plt.title('AgglomerativeC' + str(i) + ' by Cluster')
    plt.show()

In [ ]:
# create a dataframe of these two componenets
pca_df1 = pd.DataFrame(data = pca_components, columns = ['pca1', 'pca2'])
pca_df1.head()

In [ ]:
# Concatenate the clusters labels to the dataframe
pca_df1 = pd.concat([pca_df1,pd.DataFrame({'cluster1':y_AgglomerativeC})], axis = 1)
pca_df1.head()

In [ ]:
# Visualize the clusters in 2-axes plane
ax = sns.scatterplot(x="pca1", y="pca2", hue = "cluster1", data = pca_df1, palette =['red','green'])
plt.show()

In [ ]:
best_cols = ["BALANCE", "PURCHASES", "CASH_ADVANCE","CREDIT_LIMIT", "PAYMENTS", "MINIMUM_PAYMENTS",'Cluster']
sns.pairplot(df_AgglomerativeC[ best_cols ], hue="Cluster")

###  4. DBSCAN clustering algorithm
 DBSCAN stands for density-based spatial clustering of applications with noise. It's a density-based clustering algorithm.
It is able to find irregular-shaped clusters. It separates regions by areas of low-density so it can also detect outliers really well. This algorithm is better than k-means when it comes to working with oddly shaped data.

In [ ]:
df_DBScan = scaled_df.copy()

In [ ]:
outlier_percent = [] 

for eps in np.linspace(0.001,3,50): # check 50 values of epsilon between 0.001 and 3
    
    # Create Model
    dbscan = DBSCAN(eps=eps,min_samples=8)
    dbscan.fit(scaled_df)
   
    # Percentage of points that are outliers
    perc_outliers = 100 * np.sum(dbscan.labels_ == -1) / len(dbscan.labels_)
    outlier_percent.append(perc_outliers)

In [ ]:
sns.lineplot(x=np.linspace(0.001,3,50),y=outlier_percent, color='green')
plt.ylabel("Percentage of Points Classified as Outliers")
plt.xlabel("Epsilon Value");

In [ ]:
neigh = NearestNeighbors(n_neighbors=4)
nbrs = neigh.fit(scaled_df)
distances, indices = nbrs.kneighbors(scaled_df)
distances = np.sort(distances, axis=0)
distances = distances[:,1]
plt.plot(distances)

In [ ]:
Eps=[]
Min_samples=[]
num_clusters = []
best_n_clusters = 2
max_sil = 0
for i in  range(2,21):
    for j in  range(1,5):
        model_db = DBSCAN(eps=j, min_samples = i, metric='euclidean')
        model_db.fit(df_DBScan)
        labels = model_db.labels_
        n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)#cluster (-1), is not exactly part of a cluster. They are simply points that do not belong to any clusters and can be "ignored" to some extent.
        num_clusters.append(n_clusters_)
        Min_samples.append(i)
        Eps.append(j)
    
        score = silhouette_score(df_DBScan,  model_db.labels_)
        silhouette_coefficients.append(score)
        if (score > max_sil) and (n_clusters_>=2):
            max_sil = score
            best_n_clusters = n_clusters_ 
            min_sample = i
            eps = j

In [ ]:
dbscan = DBSCAN(eps=4,min_samples=2)
y_DBScan= dbscan.fit_predict(df_DBScan)

In [ ]:
labels

In [ ]:
# Checking number of items in clusters and creating 'Cluster' column
df_DBScan['Cluster'] = y_DBScan
df_DBScan['Cluster'].value_counts()

In [ ]:
# Instantiate the DBSCAN for 2 clusters
dbscan = DBSCAN(eps=4,min_samples=2)
# Fit the DBSCAN model
dbscan.fit_predict(df_DBScan)
# Calculate Silhoutte Score
score = silhouette_score(scaled_features, dbscan.labels_, metric='euclidean')
# Print the score
print('Silhouetter Average Score: %.3f' % score)

In [ ]:
describe_clusters2 = scaled_df.groupby(df_DBScan['Cluster']).mean()
describe_clusters2

In [ ]:
# create a dataframe of these two componenets
pca_df2 = pd.DataFrame(data = pca_components, columns = ['pca1', 'pca2'])
pca_df2.head()

In [ ]:
# Concatenate the clusters labels to the dataframe
pca_df2 = pd.concat([pca_df2,pd.DataFrame({'cluster2':y_DBScan})], axis = 1)
pca_df2.head()

In [ ]:
# Visualize the clusters in 2-axes plane
ax = sns.scatterplot(x="pca1", y="pca2", hue = "cluster2", data = pca_df2, palette =['red','green','blue'])
plt.show()

In [ ]:
best_cols = ["BALANCE", "PURCHASES", "CASH_ADVANCE","CREDIT_LIMIT", "PAYMENTS", "MINIMUM_PAYMENTS",'Cluster']
sns.pairplot(df_DBScan[ best_cols ], hue="Cluster")

### 5. Gaussian Mixture Models (GMM) 
A Gaussian mixture model (GMM) attempts to find a mixture of multi-dimensional Gaussian probability distributions that best model any input dataset. In the simplest case, GMMs can be used for finding clusters in the same manner as k-means, but because GMM contains a probabilistic model under the hood, it is also possible to find probabilistic cluster assignments.

The Gaussian mixture model uses multiple Gaussian distributions to fit arbitrarily shaped data.

There are several single Gaussian models that act as hidden layers in this hybrid model. So the model calculates the probability that a data point belongs to a specific Gaussian distribution and that's the cluster it will fall under.

In [ ]:
df_GMM = scaled_df.copy()

In [ ]:
def evaluate_metrics(df1,min_n_components=2, max_n_components=11, rand_state=1):

    silhouette = []
    ch_score = []
    db_score = []
    for k in range(min_n_components, max_n_components):#this range is optional
          
        gmm = GaussianMixture(n_components=k, random_state=1 )   #each gaussian in your mixture is one component 
        y_GMM = gmm.fit_predict(df_GMM)
   
        silhouette.append(silhouette_score(scaled_features, y_GMM))
        ch_score.append(calinski_harabasz_score(scaled_features, y_GMM))
        db_score.append(davies_bouldin_score(scaled_features, y_GMM))        

    
    fig, ax = plt.subplots(2, 2, figsize=(15, 10))
    ax[0][0].plot(range(min_n_components, max_n_components), silhouette, '-x', linewidth=2)
    ax[0][0].set_xlabel('No. of clusters')
    ax[0][0].set_ylabel('Silhouette Score')
    
    ax[0][1].plot(range(min_n_components, max_n_components), ch_score, '-x', linewidth=2)
    ax[0][1].set_xlabel('No. of clusters')
    ax[0][1].set_ylabel('Calinski Harabasz Score')
    
    ax[1][0].plot(range(min_n_components, max_n_components), db_score, '-x', linewidth=2)
    ax[1][0].set_xlabel('No. of clusters')
    ax[1][0].set_ylabel('Davies Bouldin Score')
    fig.suptitle('Metrics to evaluate the number of clusters')
    plt.show()

In [ ]:
evaluate_metrics(scaled_features, min_n_components=2, max_n_components=11, rand_state=1)

In [ ]:
gmm = GaussianMixture(n_components=2)
y_GMM = gmm.fit_predict(df_GMM)

In [ ]:
# Checking number of items in clusters and creating 'Cluster' column
df_GMM['Cluster'] = y_GMM
df_GMM['Cluster'].value_counts()

In [ ]:
# Instantiate the GMM for 2 clusters
gmm = GaussianMixture(n_components=2)
# Fit the GMM model
gmm.fit_predict(df_GMM)
# Calculate Silhoutte Score
score = silhouette_score(scaled_features, y_GMM, metric='euclidean')
# Print the score
print('Silhouetter Average Score: %.3f' % score)

In [ ]:
describe_clusters3 = scaled_df.groupby(df_GMM['Cluster']).mean()
describe_clusters3

In [ ]:
# create a dataframe of these two componenets
pca_df3 = pd.DataFrame(data = pca_components, columns = ['pca1', 'pca2'])
pca_df3.head()

In [ ]:
# Concatenate the clusters labels to the dataframe
pca_df3 = pd.concat([pca_df3,pd.DataFrame({'cluster3':y_GMM})], axis = 1)
pca_df3.head()

In [ ]:
# Visualize the clusters in 2-axes plane
ax = sns.scatterplot(x="pca1", y="pca2", hue = "cluster3", data = pca_df3, palette =['red','green'])
plt.show()

In [ ]:
best_cols = ["BALANCE", "PURCHASES", "CASH_ADVANCE","CREDIT_LIMIT", "PAYMENTS", "MINIMUM_PAYMENTS",'Cluster']
sns.pairplot(df_GMM[ best_cols ], hue="Cluster")

### Conclusion :
It seems that for this data set, k_Means performed better and was able to cluster better.
Using this algorithm, customers are divided into three groups as follows:

## cluster0: 
low balance, low purchase, low cash advance, low payment and low minimum payment, medium credit cart and medium purches frequency.

## cluster1:
low purchase, medium balance,cash advance,payment and minimum payment, high credit cart

## cluster2: 
low cash advance,payment, medium balance,purchase,payment, high credit cart,  purchase frequency and  purchase installment freguency.   

DB scan for this dataset seems to have had no good results.